In [6]:
!nvidia-smi

!pip -q install -U \
    transformers \
    accelerate \
    bitsandbytes \
    sentencepiece \
    safetensors \
    tqdm

Sun Jul 12 13:32:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("/content/drive/MyDrive/EmotionAwareFMD")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "affective_features"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_DIR / "liar2_finance_full.csv"

CHECKPOINT_PATH = OUTPUT_DIR / "liar2_finance_emollama_checkpoint.csv"
AUDIT_PATH = OUTPUT_DIR / "liar2_finance_emollama_raw_audit.csv"
FINAL_PATH = OUTPUT_DIR / "liar2_finance_with_affective_features.csv"

df = pd.read_csv(INPUT_PATH)

TEXT_COL = "statement"
LABEL_COL = "binary_label"

print("Input:", INPUT_PATH)
print("Rows:", len(df))
print("Output directory:", OUTPUT_DIR)

Mounted at /content/drive
Input: /content/drive/MyDrive/EmotionAwareFMD/data/liar2_finance_full.csv
Rows: 2048
Output directory: /content/drive/MyDrive/EmotionAwareFMD/outputs/affective_features


In [4]:
# Confirm final dataset size
assert len(df) == 2048, f"Expected 2,048 rows, found {len(df)}"

print("Split and label distribution:")
display(
    df.groupby(["split", "binary_label_name"])
      .size()
      .unstack(fill_value=0)
)

pair_check = (
    df.groupby("matched_pair_id")
      .agg(
          rows=("statement", "size"),
          unique_labels=("binary_label", "nunique"),
          unique_splits=("split", "nunique")
      )
)

print("\nUnique matched pairs:", df["matched_pair_id"].nunique())
print("Pairs without exactly 2 rows:", (pair_check["rows"] != 2).sum())
print("Pairs without both labels:", (pair_check["unique_labels"] != 2).sum())
print("Pairs appearing across multiple splits:", (pair_check["unique_splits"] != 1).sum())

assert df["matched_pair_id"].nunique() == 1024
assert (pair_check["rows"] == 2).all()
assert (pair_check["unique_labels"] == 2).all()
assert (pair_check["unique_splits"] == 1).all()

print("\nAll LIAR2-Finance pair and split checks passed.")

Split and label distribution:


binary_label_name,genuine,misleading
split,,
test,94,94
train,816,816
validation,114,114



Unique matched pairs: 1024
Pairs without exactly 2 rows: 0
Pairs without both labels: 0
Pairs appearing across multiple splits: 0

All LIAR2-Finance pair and split checks passed.


In [7]:
# STEP 4 — LOAD EMOLLAMA TOKENIZER (KEEP THIS CELL)

from transformers import AutoTokenizer

MODEL_NAME = "lzw1008/Emollama-chat-7b"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/934 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

Tokenizer loaded successfully.


In [8]:
# STEP 5 — LOAD EMOLLAMA IN 8-BIT (KEEP THIS CELL)

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

model.eval()

print("EmoLLaMA loaded successfully in 8-bit.")
print("Device:", next(model.parameters()).device)
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1e9, 2),
    "GB"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

EmoLLaMA loaded successfully in 8-bit.
Device: cuda:0
GPU memory allocated: 7.01 GB


In [9]:
# STEP 6 — DEFINE AFFECTIVE PROMPTS AND SCORE PARSER (KEEP THIS CELL)

import re
import numpy as np

EMOTIONS = ["anger", "fear", "joy", "sadness"]

FEATURE_COLUMNS = [
    "affect_anger",
    "affect_fear",
    "affect_joy",
    "affect_sadness",
    "affect_valence"
]


def build_emotion_prompt(text, emotion):
    text = " ".join(str(text).split())

    return f"""Human:
Task: Assign a numerical value between 0 (least {emotion}) and 1 (most {emotion}) to represent the intensity of emotion {emotion} expressed in the text.
Text: {text}
Emotion: {emotion}
Intensity Score:

Assistant:
"""


def build_valence_prompt(text):
    text = " ".join(str(text).split())

    return f"""Human:
Task: Evaluate the valence intensity of the writer's mental state based on the text, assigning it a real-valued score from 0 (most negative) to 1 (most positive).
Text: {text}
Intensity Score:

Assistant:
"""


def parse_score(response):
    numbers = re.findall(
        r"[-+]?(?:\d+(?:\.\d+)?|\.\d+)",
        str(response)
    )

    for number in numbers:
        value = float(number)

        if 0.0 <= value <= 1.0:
            return value

    return np.nan

In [10]:
# STEP 7 — DEFINE BATCHED EMOLLAMA INFERENCE (KEEP THIS CELL)

GENERATION_BATCH_SIZE = 4
MAX_INPUT_TOKENS = 256
MAX_NEW_TOKENS = 16


@torch.inference_mode()
def generate_responses(prompts, batch_size=GENERATION_BATCH_SIZE):
    responses = []

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS
        )

        inputs = {
            key: value.to("cuda")
            for key, value in inputs.items()
        }

        input_length = inputs["input_ids"].shape[1]

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

        generated_tokens = outputs[:, input_length:]

        batch_responses = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True
        )

        responses.extend(
            response.strip()
            for response in batch_responses
        )

    return responses

In [11]:
# STEP 8 — DEFINE AFFECTIVE FEATURE EXTRACTION (KEEP THIS CELL)

def extract_affective_features(records):
    prompts = []
    metadata = []

    for row_id, text in records:
        for emotion in EMOTIONS:
            prompts.append(build_emotion_prompt(text, emotion))
            metadata.append({
                "id": row_id,
                "feature": f"affect_{emotion}"
            })

        prompts.append(build_valence_prompt(text))
        metadata.append({
            "id": row_id,
            "feature": "affect_valence"
        })

    responses = generate_responses(prompts)

    audit_df = pd.DataFrame(metadata)
    audit_df["raw_response"] = responses
    audit_df["score"] = audit_df["raw_response"].apply(parse_score)
    audit_df["parse_success"] = audit_df["score"].notna()

    features_df = (
        audit_df
        .pivot(
            index="id",
            columns="feature",
            values="score"
        )
        .reindex(columns=FEATURE_COLUMNS)
        .reset_index()
    )

    features_df.columns.name = None

    return features_df, audit_df

In [17]:
# STEP 9 — RUN RESUMABLE FULL EXTRACTION (KEEP THIS CELL)

from tqdm.auto import tqdm

ROW_CHUNK_SIZE = 25

# Load existing progress if the extraction was previously interrupted.
if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH)
else:
    checkpoint_df = pd.DataFrame(columns=["id"] + FEATURE_COLUMNS)

if AUDIT_PATH.exists():
    audit_all_df = pd.read_csv(AUDIT_PATH)
else:
    audit_all_df = pd.DataFrame(
        columns=["id", "feature", "raw_response", "score", "parse_success"]
    )

# Only rows with all five valid features count as completed.
if checkpoint_df.empty:
    completed_ids = set()
else:
    completed_ids = set(
        checkpoint_df.loc[
            checkpoint_df[FEATURE_COLUMNS].notna().all(axis=1),
            "id"
        ]
    )

pending_df = df.loc[~df["id"].isin(completed_ids)].copy()

print(f"Already completed: {len(completed_ids):,}")
print(f"Remaining rows: {len(pending_df):,}")

for start in tqdm(
    range(0, len(pending_df), ROW_CHUNK_SIZE),
    desc="Extracting affective features"
):
    chunk = pending_df.iloc[start:start + ROW_CHUNK_SIZE]

    records = list(
        zip(
            chunk["id"].tolist(),
            chunk[TEXT_COL].tolist()
        )
    )

    chunk_features, chunk_audit = extract_affective_features(records)

    checkpoint_df = pd.concat(
        [checkpoint_df, chunk_features],
        ignore_index=True
    )

    checkpoint_df = (
        checkpoint_df
        .drop_duplicates(subset="id", keep="last")
        .sort_values("id")
        .reset_index(drop=True)
    )

    audit_all_df = pd.concat(
        [audit_all_df, chunk_audit],
        ignore_index=True
    )

    audit_all_df = (
        audit_all_df
        .drop_duplicates(subset=["id", "feature"], keep="last")
        .reset_index(drop=True)
    )

    # Save after every chunk so the run can be resumed.
    checkpoint_df.to_csv(CHECKPOINT_PATH, index=False)
    audit_all_df.to_csv(AUDIT_PATH, index=False)

successful_ids = set(
    checkpoint_df.loc[
        checkpoint_df[FEATURE_COLUMNS].notna().all(axis=1),
        "id"
    ]
)

remaining_df = df.loc[~df["id"].isin(successful_ids)]

print("\nExtraction pass finished.")
print("Successfully completed:", len(successful_ids))
print("Rows requiring retry:", len(remaining_df))

if remaining_df.empty:
    final_df = df.merge(
        checkpoint_df[["id"] + FEATURE_COLUMNS],
        on="id",
        how="left",
        validate="one_to_one",
        sort=False
    )

    final_df.to_csv(FINAL_PATH, index=False)

    print("Final dataset saved to:")
    print(FINAL_PATH)

Already completed: 2,047
Remaining rows: 1


Extracting affective features:   0%|          | 0/1 [00:00<?, ?it/s]


Extraction pass finished.
Successfully completed: 2047
Rows requiring retry: 1


# **Factors Finance**

In [5]:
# STEP 10 — LOAD AND VALIDATE FACTORS DATASET (KEEP THIS CELL)

FACTORS_PATH = DATA_DIR / "factors_finance_full_final.csv"

FACTORS_OUTPUT_DIR = OUTPUT_DIR
FACTORS_CHECKPOINT_PATH = (
    FACTORS_OUTPUT_DIR / "factors_emollama_checkpoint.csv"
)
FACTORS_AUDIT_PATH = (
    FACTORS_OUTPUT_DIR / "factors_emollama_raw_audit.csv"
)
FACTORS_FINAL_PATH = (
    FACTORS_OUTPUT_DIR / "factors_with_affective_features.csv"
)

factors_df = pd.read_csv(FACTORS_PATH)

FACTORS_TEXT_COL = "text"
FACTORS_LABEL_COL = "label_id"
FACTORS_PAIR_COL = "pair_id"

assert len(factors_df) == 522
assert factors_df[FACTORS_TEXT_COL].notna().all()
assert factors_df[FACTORS_TEXT_COL].astype(str).str.strip().ne("").all()
assert factors_df["id"].is_unique
assert set(factors_df["label"].unique()) == {"genuine", "misleading"}
assert set(factors_df[FACTORS_LABEL_COL].unique()) == {0, 1}

pair_check = (
    factors_df.groupby(FACTORS_PAIR_COL)
    .agg(
        rows=("id", "size"),
        unique_labels=("label", "nunique"),
    )
)

assert factors_df[FACTORS_PAIR_COL].nunique() == 261
assert (pair_check["rows"] == 2).all()
assert (pair_check["unique_labels"] == 2).all()

print("FACTors dataset validated.")
print("Rows:", len(factors_df))
print("Matched pairs:", factors_df[FACTORS_PAIR_COL].nunique())
print("\nLabel distribution:")
print(factors_df["label"].value_counts())

FACTors dataset validated.
Rows: 522
Matched pairs: 261

Label distribution:
label
genuine       261
misleading    261
Name: count, dtype: int64


In [16]:
# STEP 11 — RUN RESUMABLE FACTORS EXTRACTION (KEEP THIS CELL)

from tqdm.auto import tqdm

ROW_CHUNK_SIZE = 25

if FACTORS_CHECKPOINT_PATH.exists():
    factors_checkpoint_df = pd.read_csv(FACTORS_CHECKPOINT_PATH)
else:
    factors_checkpoint_df = pd.DataFrame(
        columns=["id"] + FEATURE_COLUMNS
    )

if FACTORS_AUDIT_PATH.exists():
    factors_audit_df = pd.read_csv(FACTORS_AUDIT_PATH)
else:
    factors_audit_df = pd.DataFrame(
        columns=[
            "id",
            "feature",
            "raw_response",
            "score",
            "parse_success",
        ]
    )

if factors_checkpoint_df.empty:
    completed_ids = set()
else:
    completed_ids = set(
        factors_checkpoint_df.loc[
            factors_checkpoint_df[FEATURE_COLUMNS]
            .notna()
            .all(axis=1),
            "id",
        ]
    )

pending_df = factors_df.loc[
    ~factors_df["id"].isin(completed_ids)
].copy()

print(f"Already completed: {len(completed_ids):,}")
print(f"Remaining rows: {len(pending_df):,}")

for start in tqdm(
    range(0, len(pending_df), ROW_CHUNK_SIZE),
    desc="Extracting FACTors affective features",
):
    chunk = pending_df.iloc[
        start : start + ROW_CHUNK_SIZE
    ]

    records = list(
        zip(
            chunk["id"].tolist(),
            chunk[FACTORS_TEXT_COL].tolist(),
        )
    )

    chunk_features, chunk_audit = (
        extract_affective_features(records)
    )

    factors_checkpoint_df = pd.concat(
        [factors_checkpoint_df, chunk_features],
        ignore_index=True,
    )

    factors_checkpoint_df = (
        factors_checkpoint_df
        .drop_duplicates(subset="id", keep="last")
        .sort_values("id")
        .reset_index(drop=True)
    )

    factors_audit_df = pd.concat(
        [factors_audit_df, chunk_audit],
        ignore_index=True,
    )

    factors_audit_df = (
        factors_audit_df
        .drop_duplicates(
            subset=["id", "feature"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    factors_checkpoint_df.to_csv(
        FACTORS_CHECKPOINT_PATH,
        index=False,
    )

    factors_audit_df.to_csv(
        FACTORS_AUDIT_PATH,
        index=False,
    )

successful_ids = set(
    factors_checkpoint_df.loc[
        factors_checkpoint_df[FEATURE_COLUMNS]
        .notna()
        .all(axis=1),
        "id",
    ]
)

remaining_df = factors_df.loc[
    ~factors_df["id"].isin(successful_ids)
]

print("\nExtraction pass finished.")
print("Successfully completed:", len(successful_ids))
print("Rows requiring retry:", len(remaining_df))

if remaining_df.empty:
    factors_final_df = factors_df.merge(
        factors_checkpoint_df[
            ["id"] + FEATURE_COLUMNS
        ],
        on="id",
        how="left",
        validate="one_to_one",
        sort=False,
    )

    factors_final_df.to_csv(
        FACTORS_FINAL_PATH,
        index=False,
    )

    print("Final FACTors dataset saved to:")
    print(FACTORS_FINAL_PATH)

Already completed: 521
Remaining rows: 1


Extracting FACTors affective features:   0%|          | 0/1 [00:00<?, ?it/s]


Extraction pass finished.
Successfully completed: 521
Rows requiring retry: 1


In [18]:
# ONE-OFF REPAIR ONLY — DELETE THIS CELL AFTER THE FINAL CSV IS SAVED

FAILED_ID = 70511
text = factors_df.loc[
    factors_df["id"] == FAILED_ID,
    FACTORS_TEXT_COL
].iloc[0]

failed_emotions = {
    "affect_anger": "anger",
    "affect_joy": "joy",
}

repair_results = []

for feature, emotion in failed_emotions.items():
    prompt = build_emotion_prompt(text, emotion)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to("cuda")

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            min_new_tokens=2,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        output[0][input_length:],
        skip_special_tokens=True,
    ).strip()

    score = parse_score(response)

    print(feature, "->", repr(response), "->", score)

    repair_results.append(
        {
            "id": FAILED_ID,
            "feature": feature,
            "raw_response": response,
            "score": score,
            "parse_success": not np.isnan(score),
        }
    )

assert all(
    result["parse_success"]
    for result in repair_results
), "A repaired output is still invalid."

for result in repair_results:
    factors_checkpoint_df.loc[
        factors_checkpoint_df["id"] == FAILED_ID,
        result["feature"],
    ] = result["score"]

    audit_mask = (
        (factors_audit_df["id"] == FAILED_ID)
        & (factors_audit_df["feature"] == result["feature"])
    )

    factors_audit_df.loc[
        audit_mask,
        "raw_response",
    ] = result["raw_response"]

    factors_audit_df.loc[
        audit_mask,
        "score",
    ] = result["score"]

    factors_audit_df.loc[
        audit_mask,
        "parse_success",
    ] = True

factors_checkpoint_df.to_csv(
    FACTORS_CHECKPOINT_PATH,
    index=False,
)

factors_audit_df.to_csv(
    FACTORS_AUDIT_PATH,
    index=False,
)

assert len(factors_checkpoint_df) == 522
assert factors_checkpoint_df[
    FEATURE_COLUMNS
].notna().all().all()

factors_final_df = factors_df.merge(
    factors_checkpoint_df[
        ["id"] + FEATURE_COLUMNS
    ],
    on="id",
    how="left",
    validate="one_to_one",
    sort=False,
)

assert len(factors_final_df) == 522
assert factors_final_df[
    FEATURE_COLUMNS
].notna().all().all()

factors_final_df.to_csv(
    FACTORS_FINAL_PATH,
    index=False,
)

print("\nAll 522 FACTors rows completed.")
print("Final FACTors dataset saved to:")
print(FACTORS_FINAL_PATH)

affect_anger -> '0.438' -> 0.438
affect_joy -> '0.22' -> 0.22

All 522 FACTors rows completed.
Final FACTors dataset saved to:
/content/drive/MyDrive/EmotionAwareFMD/outputs/affective_features/factors_with_affective_features.csv
